In [1]:
import os
import gzip
import itertools
import pathlib
from collections import namedtuple
import importlib
import pickle
import pysam

import helpers.consistency as cons
import helpers.allele_db as db

importlib.reload(cons)
importlib.reload(db)

<module 'helpers.allele_db' from '/pbi/flash/edolzhenko/2024/Q2/PlatinumPedigree/pipelines/tandem-repeats/helpers/allele_db.py'>

In [2]:
inheritance_vector_path = "/home/edolzhenko/flash/2024/Q2/palladium/000_ivecs.grch38.csv"

In [3]:
ivecs = cons.load_ivecs(inheritance_vector_path)

In [4]:
def get_ivec(inheritance_vecs, trid):
    chrom, start, end = trid.split("_")[:3]
    start, end = int(start), int(end)
    for inheritance_vec in inheritance_vecs:
        if inheritance_vec["chrom"] == chrom and inheritance_vec["start"] <= start and end <= inheritance_vec["end"]:
            return inheritance_vec
    return None

In [5]:
Summary = namedtuple("Summary", "trid A B C D is_consistent mean_purity mean_len mean_gc distinct_alleles")

with open("output/summary.pkl", "rb") as file:
    tr_summaries = pickle.load(file)

In [6]:
tr_summaries[0]

Summary(trid='chr10_100000859_100000887_trsolve_A', A='AAAAAAAAAAAAAAAAAAAAAAAAAAAAA', B='AAAAAAAAAAAAAAAAAAACAAAAAAAA', C='AAAAAAAAAAAAAAAAAAAACAAAAAAAA', D='AAAAAAAAAAAAAAAAAAAAAAAAAAAAA', is_consistent=False, mean_purity=0.9825000000000002, mean_len=28.75, mean_gc=0.017549261083743842, distinct_alleles=3)

In [7]:
header = "##fileformat=VCFv4.2"
samples = ["NA12887", "NA12886", "NA12885", "NA12884", "NA12883", "NA12882", "NA12881", "NA12879", "NA12877", "NA12878"]
fields = ["CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO", "FORMAT"] + samples

In [8]:
VcfRec = namedtuple("VcfRec", "chrom pos trid end motif ref alts samples_to_alleles")


def encode_vcf_rec(rec):
    alts = ",".join(rec.alts) if rec.alts else "."
    #         CHROM        POS       ID  REF        ALT  QUAL FILTER INFO
    encoding = f"{rec.chrom}\t{rec.pos}\t.\t{rec.ref}\t{alts}\t0\t.\tTRID={rec.trid};END={rec.end}"
    #  FORMAT
    encoding += "\tGT"
    # SAMPLE
    seqs = [rec.ref] + rec.alts
    for sample in samples:
        alleles = rec.samples_to_alleles[sample]
        gt = sorted([seqs.index(a) for a in alleles])
        gt = "/".join([str(s) for s in gt])
        encoding += "\t" + gt
    return encoding


def get_vcf():
    genome = pysam.FastaFile("/home/edolzhenko/common/hg38/hg38.fa")

    yield header
    yield "#" + "\t".join(fields)
    for summary in tr_summaries:
        if not summary.is_consistent:
            continue
        trid = summary.trid
        ivec = get_ivec(ivecs, trid)
        chrom, start, end, source, motif = trid.split("_")
        start, end = int(start), int(end)
        ref = genome.fetch(chrom, start, end).upper()
        alts = list(set(val for val in [summary.A, summary.B, summary.C, summary.D] if val and val != ref))

        samples_to_alleles = {}
        for sample in samples:
            haps = ivec[sample]
            alleles = [getattr(summary, hap) for hap in haps]
            samples_to_alleles[sample] = alleles

        rec = VcfRec(chrom=chrom, pos=start + 1, trid=trid, end=end,
                     motif=motif, ref=ref, alts=alts, samples_to_alleles=samples_to_alleles) # <- fix motif setting!!!
        yield encode_vcf_rec(rec)


with open("scratch/tr_calls.vcf", "w") as file:
    for line in get_vcf():
        print(line, file=file)

In [9]:
%%bash

zcat < /home/edolzhenko/flash/2024/Q1/trgt-bench/201_run_trgt/output/HG002.vcf.gz | grep "##" | grep -v "##trgt" > scratch/header.txt
cat scratch/tr_calls.vcf | grep "#" | grep -v "##" >> scratch/header.txt

In [10]:
%%bash

bcftools="/pbi/flash/smrtlink/smrtlink-develop/smrtlink/current/bundles/smrttools/current/private/otherbins/all/bin/bcftools"

$bcftools reheader --header scratch/header.txt --output output/tr_calls.vcf scratch/tr_calls.vcf
$bcftools sort output/tr_calls.vcf -Oz -o output/tr_calls.vcf.gz

Writing to /tmp/bcftools.o9fDYa
Merging 1 temporary files
Cleaning
Done


In [11]:
%%bash

/pbi/flash/edolzhenko/2024/Q2/palladium/002_tools/vcf_validator_linux < output/tr_calls.vcf.gz

[info] Reading from standard input...
[debug] Detected .gz compression


Lines read: 10000
Lines read: 20000
Lines read: 30000
Lines read: 40000
Lines read: 50000
Lines read: 60000
Lines read: 70000
Lines read: 80000
Lines read: 90000
Lines read: 100000
Lines read: 110000
Lines read: 120000
Lines read: 130000
Lines read: 140000
Lines read: 150000
Lines read: 160000
Lines read: 170000
Lines read: 180000
Lines read: 190000
Lines read: 200000
Lines read: 210000
Lines read: 220000
Lines read: 230000
Lines read: 240000
Lines read: 250000
Lines read: 260000
Lines read: 270000
Lines read: 280000
Lines read: 290000
Lines read: 300000
Lines read: 310000
Lines read: 320000
Lines read: 330000
Lines read: 340000
Lines read: 350000
Lines read: 360000
Lines read: 370000
Lines read: 380000
Lines read: 390000
Lines read: 400000
Lines read: 410000
Lines read: 420000
Lines read: 430000
Lines read: 440000
Lines read: 450000
Lines read: 460000
Lines read: 470000
Lines read: 480000
Lines read: 490000
Lines read: 500000
Lines read: 510000
Lines read: 520000
Lines read: 530000


[info] Summary report written to : stdin.errors_summary.1717703063236.txt
[info] According to the VCF specification, the input file is valid


In [12]:
%%bash

zcat output/tr_calls.vcf.gz | grep -v "#" | head -n 3

chr1	759447	.	ACACACACAAACACACACACACACACACACACACACACACACACACACACAC	ACACACACAAACACACACACACACACACACACACACACACACACACACACACAC,ACACACACAAACACACACACACACACACACACACACACACACACACACCC	0	.	TRID=chr1_759446_759498_trsolve_AC;END=759498	GT	1/2	1/2	1/2	2/2	2/2	1/2	1/2	2/2	1/2	2/2
chr1	814179	.	ACCAAAACCAAAAC	ACCAACACCAAAAC	0	.	TRID=chr1_814178_814192_trsolve_ACCAAA;END=814192	GT	0/0	0/0	0/0	0/1	0/1	0/0	0/0	0/1	0/1	0/0
chr1	816109	.	AAAAAAAAAAAAA	AAAAAAAAAAAAAA	0	.	TRID=chr1_816108_816121_trsolve_A;END=816121	GT	0/1	0/1	0/1	0/0	0/0	0/1	0/1	0/0	0/1	0/0


In [34]:
total_delta = 0
na12878_path = "/pbi/flash/tmokveld/data/AWS_sync/PB_TRs/TRGT/GRCh38_v1.0_50bp_merge/493ef25/2188_DM_GRCh38_50bp_merge.sorted.vcf.gz"
with gzip.open(na12878_path, "r") as file:
    for line in file:
        line = line.decode("ascii")
        if line[0] == "#":
            continue
        chrom, start, qual, ref, alts, _, _, locus_info, _, sample_info = line.split()
        gt = sample_info.split(":")[0]
        if gt == ".":
            continue
        alleles = [ref]
        if alts != ".":
            alleles.extend(alts.split(","))
        delta = sum(abs(len(alleles[int(i)]) - len(alleles[0])) for i in gt.split("/"))
        total_delta += delta

total_delta

17053647

In [36]:
17053647 / 10**6

17.053647